https://www.amazon.com.br/gp/bestsellers/?ref_=nav_em_cs_bestsellers_0_1_1_2

https://www.selenium.dev/documentation/

In [67]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import pandas as pd
import random
import requests
import os

**Como Encontrar Elementos do HTML**  

- `find_element(By.ID, "id")`  
- `find_element(By.NAME, "name")`  
- `find_element(By.XPATH, "xpath")`  
- `find_element(By.LINK_TEXT, "link text")`  
- `find_element(By.PARTIAL_LINK_TEXT, "partial link text")`  
- **`find_element(By.TAG_NAME, "tag name")`**  
- **`find_element(By.CLASS_NAME, "class name")`**  
- `find_element(By.CSS_SELECTOR, "css selector")`

In [52]:
service = Service()

options = webdriver.ChromeOptions()

driver = webdriver.Chrome(service=service, options=options)

In [53]:
url = 'https://www.amazon.com.br/gp/bestsellers/?ref_=nav_em_cs_bestsellers_0_1_1_2'

driver.get(url)

### PEGANDO O LINK DE CADA PRODUTO

In [54]:
links = driver.find_elements(By.CLASS_NAME, "a-link-normal")
links_produtos = [link.get_attribute('href') for link in links if "/dp/" in link.get_attribute('href')]
links_produtos = list(set(links_produtos))

### TENTANDO PEGAR O NOME DO PRODUTO PARA CADA LISTA DE LINK

In [57]:
nomes = []
notas = []
qntAvaliacoes = []
prices = []
urls = []
for l in links_produtos:
    driver.get(l)
    # PEGA O NOME
    nome = driver.find_element(By.ID, 'productTitle')
    nomes.append(nome.text)

    # PEGA A NOTA DE AVALIACAO
    nota = driver.find_element(By.CSS_SELECTOR, "span[data-hook='rating-out-of-text']")
    texto_nota = nota.text.strip()  # Extrair o texto e remover espaços em branco
    # Extrair apenas a nota (exemplo: "4,6" de "4,6 de 5")
    nota = texto_nota.split(" de ")[0]  # Divide o texto e pega a primeira parte
    notas.append(nota)

    # PEGA A QUANTIDADE DE AVALIACAO
    qntAvaliacao = driver.find_element(By.ID, 'acrCustomerReviewText')
    qntAvaliacoes.append(qntAvaliacao.text)

    # PEGANDO O PREÇO DO PRODUTO
    preco = driver.find_element(By.CLASS_NAME, "a-price-whole")
    prices.append(f'R${preco.text}')

    # pegando a url
    image_element = driver.find_element(By.CSS_SELECTOR, "img#landingImage")
    url_imagem = image_element.get_attribute('src')
    urls.append(url_imagem)

### FAZENDO O DF

In [58]:
infos = {'Produto': nomes,
         'URL': links_produtos,
         'Nota': notas,
         'N de avaliações': qntAvaliacoes,
         'Preco': prices,
         'URL_IMG': urls,
         'tweet_text': None,
         'media_path': None}

In [59]:
df = pd.DataFrame(infos)

In [68]:
def limpa_texto(texto, n):
    n = n
    palavras = texto.split()
    texto_limpo = ' '.join(palavras[:n])

    if texto_limpo[:-1] == ',':
        texto_limpo = texto_limpo.rstrip(',')

    return texto_limpo

In [ ]:
def gerar_msg(row):
    mensagens_aleatorias = random.choice([
    "Confira esta oferta incrível: ",
    "Não perca esta promoção: ",
    "Olha só o que encontrei: ",
    "Grande oportunidade: ",
    "Promoção imperdível: "
    ])
    return f"{mensagens_aleatorias}{row['Produto']} com mais de {row['N de avaliações']} e nota {row['Nota']} por apenas {row['Preco']}! \n\nLink do produto: {row['URL']}"

In [70]:
# Função para baixar e salvar a imagem
def baixar_imagem(url, nome_arquivo):
    try:
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(nome_arquivo, 'wb') as file:
                for chunk in response.iter_content(1024):
                    file.write(chunk)
            return True
        else:
            print(f"Erro ao baixar a imagem: {url}")
            return False
    except Exception as e:
        print(f"Erro ao baixar a imagem: {e}")
        return False

In [71]:
df['Produto'] = df['Produto'].apply(limpa_texto, n=8)

In [72]:
df['tweet_text'] = df.apply(gerar_msg, axis=1)

In [73]:
# Atualizar o DataFrame com o caminho da imagem
for index, row in df.iterrows():
    # Extrair o nome do produto para usar como nome do arquivo
    nome_produto = row['Produto'].replace('/', '_').replace('\\', '_').replace(':', '_').replace('*', '_').replace('?', '_').replace('"', '_').replace('<', '_').replace('>', '_').replace('|', '_')
    nome_arquivo = f"media/{nome_produto}.jpg"  # Assume que a imagem é JPG

    # Baixar a imagem (substitua 'URL_DA_IMAGEM' pelo campo correto do seu DataFrame)
    # Aqui, estou assumindo que você tem um campo 'URL_DA_IMAGEM' no DataFrame.
    # Se não tiver, você precisará obter a URL da imagem de outra forma.
    url_imagem = row['URL_IMG']  # Substitua pelo campo correto ou obtenha a URL de outra forma

    if baixar_imagem(url_imagem, nome_arquivo):
        df.at[index, 'media_path'] = nome_arquivo
    else:
        df.at[index, 'media_path'] = None  # Ou algum valor padrão

In [74]:
df.to_csv('produtos - Página1.csv', index=False)

In [75]:
df.head(5)

,Produto,URL,Nota,N de avaliações,Preco,URL_IMG,tweet_text,media_path
0,Base De Carregamento Do Dualsense-padrão-plays...,https://www.amazon.com.br/Base-Carregamento-Do...,"4,9",11.579 avaliações de clientes,R$189,https://m.media-amazon.com/images/I/41xfd4g3PZ...,Não perca esta promoção: Base De Carregamento ...,media/Base De Carregamento Do Dualsense-padrão...
1,"Havit HV-H2232d - Fone de Ouvido, Gamer, Ilumi...",https://www.amazon.com.br/Headphone-HV-H2232d-...,"4,5",7.218 avaliações de clientes,R$87,https://m.media-amazon.com/images/I/516HwNOJoU...,Promoção imperdível: Havit HV-H2232d - Fone de...,"media/Havit HV-H2232d - Fone de Ouvido, Gamer,..."
2,Echo Pop | Smart speaker compacto com som,https://www.amazon.com.br/Echo-Pop-Cor-Preta/d...,"4,8",59.263 avaliações de clientes,R$360,https://m.media-amazon.com/images/I/71eWKwcVjn...,Não perca esta promoção: Echo Pop | Smart spea...,media/Echo Pop _ Smart speaker compacto com so...
3,"Epson EcoTank L3250 - Multifuncional, Tanque d...",https://www.amazon.com.br/Multifuncional-Epson...,"4,8",22.542 avaliações de clientes,R$1.119,https://m.media-amazon.com/images/I/51T-6cdhya...,Não perca esta promoção: Epson EcoTank L3250 -...,"media/Epson EcoTank L3250 - Multifuncional, Ta..."
4,Kit Cueca Boxer Polo Wear 12 Peças Lisas,https://www.amazon.com.br/cuecas-Polo-Wear-Mas...,"4,4",8.740 avaliações de clientes,R$108,https://m.media-amazon.com/images/I/51436Fn1So...,Confira esta oferta incrível: Kit Cueca Boxer ...,media/Kit Cueca Boxer Polo Wear 12 Peças Lisas...
